# Telco Customer Churn Analysis
**Role**: Senior Data Analyst  
**Objective**: Build an end-to-end analytics workflow in Python to ingest, clean, explore, and export the Telco Customer Churn dataset. The insights generated here will feed downstream dashboards in SQL, Excel, and Power BI.

### Analytics Objectives:
1. **Data Cleaning**: Handle empty values in `TotalCharges`, drop duplicate rows, and standardize string columns.
2. **Feature Engineering**: Bin customers into tenure groups, map categories to binary flags, and count services.
3. **Exploratory Data Analysis**: Evaluate customer demographics, contract models, internet services, and billing amounts.
4. **Visual Analytics**: Output correlation heatmaps and churn distribution rates.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual parameters
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Libraries successfully imported!")

## 1. Data Ingestion & Initial Inspection

In [ ]:
raw_path = r"../data/raw/telco_customer_churn.csv"
df = pd.read_csv(raw_path)
print(f"Raw dataset shape: {df.shape}")
df.info()
df.head()

## 2. Data Cleaning
- Check and remove duplicates.
- Address the blank spaces in `TotalCharges`. Convert to numeric float and fill NaNs (since tenure is 0 for these records, we set TotalCharges to 0.0).

In [ ]:
# Check duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
if duplicates > 0:
    df.drop_duplicates(inplace=True)

# Inspect blank spaces in TotalCharges
blanks_count = (df['TotalCharges'] == ' ').sum()
print(f"Blank spaces in TotalCharges: {blanks_count}")

# Convert TotalCharges to float, fill missing values with 0.0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# Strip whitespace from object columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip()

print("\nMissing values count after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Feature Engineering
- **TenureGroup**: Group tenure months into logical bins.
- **TotalInternetServices**: Count of security/streaming services.
- **Numeric mappings**: Convert binary classifications ('Yes'/'No') to 1/0 integers.

In [ ]:
# Map binary targets to 1/0
df['ChurnNumeric'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)
df['PartnerNumeric'] = df['Partner'].apply(lambda x: 1 if x == 'Yes' else 0)
df['DependentsNumeric'] = df['Dependents'].apply(lambda x: 1 if x == 'Yes' else 0)
df['PhoneServiceNumeric'] = df['PhoneService'].apply(lambda x: 1 if x == 'Yes' else 0)
df['PaperlessBillingNumeric'] = df['PaperlessBilling'].apply(lambda x: 1 if x == 'Yes' else 0)

# Bin tenure
def bin_tenure(months):
    if months <= 12: return "0-12 Month"
    elif months <= 24: return "12-24 Month"
    elif months <= 48: return "24-48 Month"
    elif months <= 60: return "48-60 Month"
    else: return "60+ Month"

df['TenureGroup'] = df['tenure'].apply(bin_tenure)
df['SeniorCitizenLabel'] = df['SeniorCitizen'].apply(lambda x: 'Senior' if x == 1 else 'Non-Senior')

# Count active internet services
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalInternetServices'] = df[services].apply(lambda r: sum(1 for c in r if c == 'Yes'), axis=1)
df['HasInternetService'] = df['InternetService'].apply(lambda x: 1 if x != 'No' else 0)

df.head(2)

## 4. Exploratory Data Analysis & Visualizations

In [ ]:
# 1. Churn Rate Pie Chart
plt.figure(figsize=(6, 6))
churn_counts = df['Churn'].value_counts()
plt.pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', colors=['#4C72B0', '#C44E52'], startangle=90)
plt.title("Overall Churn Rate")
plt.show()

# 2. Churn by Contract Type
contract_churn = df.groupby(['Contract', 'Churn']).size().unstack(fill_value=0)
contract_churn_pct = contract_churn.div(contract_churn.sum(axis=1), axis=0) * 100
contract_churn_pct.plot(kind='bar', stacked=True, color=['#4C72B0', '#C44E52'])
plt.title("Churn Rate by Contract Type")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)
plt.show()

## 5. Correlation Analysis

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'ChurnNumeric', 
                'PartnerNumeric', 'DependentsNumeric', 'TotalInternetServices', 'HasInternetService']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.show()

## 6. Saving processed dataset

In [ ]:
processed_out = r"../data/processed/telco_customer_churn_cleaned.csv"
df.to_csv(processed_out, index=False, encoding='utf-8')
print(f"Notebook pipeline completed! Cleaned rows saved: {len(df)}")